Silver Layer - Binance Crypto Cleaned & Transformed

In [0]:
storage_account = "deproj1adls"
container = "crypto-data"

STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope="binance-secrets", key="storage-account-key")
spark.conf.set(f"fs.azure.account.key.{storage_account}.blob.core.windows.net", STORAGE_ACCOUNT_KEY)


BRONZE_TABLE_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/bronze/binance_crypto_raw_v3"
CHECKPOINT_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/checkpoints/silver_binance"
SILVER_TABLE_PATH = f"wasbs://{container}@{storage_account}.blob.core.windows.net/silver/binance_trades_clean"
TRIGGER_INTERVAL = "15 seconds"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
bronze_stream = (
    spark.readStream
    .format("delta")
    .load(BRONZE_TABLE_PATH)
)

bronze_stream.printSchema()

In [0]:
silver_stream = (
    bronze_stream
    .filter(col("event_type") == "trade")
    .filter(col("symbol").isNotNull())
    .filter(col("trade_price").isNotNull())
    .filter(col("trade_quantity").isNotNull())
    .withColumn("trade_price_numeric", col("trade_price").cast("double"))
    .withColumn("trade_quantity_numeric", col("trade_quantity").cast("double"))
    .withColumn("trade_value_usd", col("trade_price_numeric") * col("trade_quantity_numeric"))
    .withColumn("trade_timestamp", from_unixtime(col("trade_time") / 1000).cast("timestamp"))
    .withColumn("event_timestamp", from_unixtime(col("event_time") / 1000).cast("timestamp"))
    .withColumn("trade_date", to_date(col("trade_timestamp")))
    .withColumn("trade_hour", hour(col("trade_timestamp")))
    .withColumn("base_currency", regexp_extract(col("symbol"), "^([A-Z]+)", 1))
    .withColumn("quote_currency", regexp_extract(col("symbol"), "([A-Z]+)$", 1))
    .withColumn("silver_processing_time", current_timestamp())
    .select(
        col("trade_id"),
        col("symbol"),
        col("base_currency"),
        col("quote_currency"),
        col("trade_timestamp"),
        col("event_timestamp"),
        col("trade_date"),
        col("trade_hour"),
        col("trade_price_numeric").alias("price"),
        col("trade_quantity_numeric").alias("quantity"),
        col("trade_value_usd"),
        col("is_buyer_maker"),
        col("buyer_order_id"),
        col("seller_order_id"),
        col("bronze_ingestion_time"),
        col("silver_processing_time")
    )
)

In [0]:
silver_query = (
    silver_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start(SILVER_TABLE_PATH)
)

print(f"Silver stream started: {silver_query.id}")

In [0]:
import time

print(f"Is Active: {silver_query.isActive}")
print("\nMonitoring silver layer...")

for i in range(200):
    time.sleep(3)
    status = silver_query.status
    print(f"\nBatch {i+1}: {status['message']}")
    if 'numInputRows' in status:
        print(f"  Input Rows: {status['numInputRows']}")

In [0]:
silver_df = spark.read.format("delta").load(SILVER_TABLE_PATH)

quality_metrics = silver_df.select(
    count("*").alias("total_trades"),
    countDistinct("symbol").alias("unique_symbols"),
    countDistinct("trade_date").alias("unique_dates"),
    round(avg("price"), 2).alias("avg_price"),
    round(sum("trade_value_usd"), 2).alias("total_volume_usd"),
    min("trade_timestamp").alias("earliest_trade"),
    max("trade_timestamp").alias("latest_trade")
)

display(quality_metrics)

In [0]:
silver_query.stop()

In [0]:
%sql
SELECT 
    symbol,
    COUNT(*) as trade_count,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(MIN(price), 2) as min_price,
    ROUND(MAX(price), 2) as max_price,
    ROUND(SUM(trade_value_usd), 2) as total_volume_usd
FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/silver/binance_trades_clean`
WHERE trade_date = CURRENT_DATE()
GROUP BY symbol
ORDER BY total_volume_usd DESC

In [0]:
%sql
SELECT 
    date_trunc('hour', trade_timestamp) as hour,
    symbol,
    COUNT(*) as trades,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(SUM(trade_value_usd), 2) as volume_usd
FROM delta.`wasbs://crypto-data@deproj1adls.blob.core.windows.net/silver/binance_trades_clean`
WHERE trade_timestamp >= current_timestamp() - INTERVAL 24 HOURS
GROUP BY date_trunc('hour', trade_timestamp), symbol
ORDER BY hour DESC, volume_usd DESC
LIMIT 50